# Bonus Unit 1: Let's train Huggy the Dog 🐶 to fetch a stick

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit2/thumbnail.png" alt="Bonus Unit 1Thumbnail">

In this notebook, we'll reinforce what we learned in the first Unit by **teaching Huggy the Dog to fetch the stick and then play with it directly in your browser**

⬇️ Here is an example of what **you will achieve at the end of the unit.** ⬇️ (launch ▶ to see)

In [1]:
%%html
<video controls autoplay><source src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy.mp4" type="video/mp4"></video>

### The environment 🎮

- Huggy the Dog, an environment created by [Thomas Simonini](https://twitter.com/ThomasSimonini) based on [Puppo The Corgi](https://blog.unity.com/technology/puppo-the-corgi-cuteness-overload-with-the-unity-ml-agents-toolkit)

### The library used 📚

- [MLAgents](https://github.com/Unity-Technologies/ml-agents)

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the Github Repo](https://github.com/huggingface/deep-rl-class/issues).

## Objectives of this notebook 🏆

At the end of the notebook, you will:

- Understand **the state space, action space and reward function used to train Huggy**.
- **Train your own Huggy** to fetch the stick.
- Be able to play **with your trained Huggy directly in your browser**.




## This notebook is from Deep Reinforcement Learning Course
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

In this free course, you will:

- 📖 Study Deep Reinforcement Learning in **theory and practice**.
- 🧑‍💻 Learn to **use famous Deep RL libraries** such as Stable Baselines3, RL Baselines3 Zoo, CleanRL and Sample Factory 2.0.
- 🤖 Train **agents in unique environments**

And more check 📚 the syllabus 👉 https://simoninithomas.github.io/deep-rl-course

Don’t forget to **<a href="http://eepurl.com/ic5ZUD">sign up to the course</a>** (we are collecting your email to be able to **send you the links when each Unit is published and give you information about the challenges and updates).**


The best way to keep in touch is to join our discord server to exchange with the community and with us 👉🏻 https://discord.gg/ydHrjt3WP5

## Prerequisites 🏗️

Before diving into the notebook, you need to:

🔲 📚 **Develop an understanding of the foundations of Reinforcement learning** (MC, TD, Rewards hypothesis...) by doing Unit 1

🔲 📚 **Read the introduction to Huggy** by doing Bonus Unit 1

## Local runtime requirements

This adapted notebook uses Huggy's prebuilt **Linux x86_64** Unity environment. It expects the isolated Python 3.10.12 environment described in [`../README.md`](../README.md). CPU training is supported, though it may take longer than the course's 30–45 minute estimate. The [course currently recommends Google Colab](https://huggingface.co/learn/deep-rl-course/unitbonus1/train) for the original hands-on workflow.


## Check the local ML-Agents environment

The next cell checks the Python version, ML-Agents packages, and command-line tools before downloading the Unity environment.


In [2]:
from importlib.metadata import PackageNotFoundError, version
import os
import platform
import shutil
import stat
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if candidate.name == "Notebook" and (candidate / "notebooks" / "bonus_unit1.ipynb").is_file():
            return candidate
        if candidate.name == "notebooks" and (candidate / "bonus_unit1.ipynb").is_file():
            return candidate.parent
        nested = candidate / "Notebook"
        if (nested / "notebooks" / "bonus_unit1.ipynb").is_file():
            return nested
    raise RuntimeError("Could not find Bonus Unit 1/Notebook. Start Jupyter from that folder.")


NOTEBOOK_DIR = find_notebook_dir()
RUNS_DIR = NOTEBOOK_DIR / "runs"
ENVIRONMENT_DIR = RUNS_DIR / "environment"
HUGGY_LINUX_DIR = ENVIRONMENT_DIR / "linux"
RESULTS_DIR = RUNS_DIR / "results"
CONFIG_PATH = NOTEBOOK_DIR / "config" / "Huggy.yaml"
HUGGY_ARCHIVE = ENVIRONMENT_DIR / "Huggy.zip"
HUGGY_EXECUTABLE = HUGGY_LINUX_DIR / "Huggy" / "Huggy.x86_64"
RUN_ID = os.environ.get("HUGGY_RUN_ID", "Huggy")

if platform.system() != "Linux" or platform.machine().lower() not in {"x86_64", "amd64"}:
    raise RuntimeError("The bundled Huggy environment requires Linux x86_64.")
if tuple(sys.version_info[:3]) != (3, 10, 12):
    current_python = sys.version.split()[0]
    raise RuntimeError(
        f"This notebook requires Python 3.10.12; the active Jupyter kernel is Python {current_python}. "
        "The notebook is using the wrong kernel. From 'Bonus Unit 1', create it with `conda env create -f environment.yml` (or update an existing environment with `conda env update -n huggingface-bonus-unit1 -f environment.yml --prune`), then activate it:\n"
        "  conda env create -f environment.yml\n"
        "  conda activate huggingface-bonus-unit1\n"
        "  python -m ipykernel install --user --name bonus-unit1 --display-name 'Python 3.10 (Huggy)'\n"
        "Then select Kernel > Change Kernel > Python 3.10 (Huggy), restart, and run again. See Notebook/README.md."
    )

missing = []
for distribution in ("mlagents", "mlagents-envs"):
    try:
        installed_version = version(distribution)
    except PackageNotFoundError:
        missing.append(f"{distribution} is not installed")
    else:
        if installed_version != "1.1.0":
            missing.append(f"{distribution}==1.1.0 is required (found {installed_version})")
for command in ("mlagents-learn", "mlagents-push-to-hf"):
    if shutil.which(command) is None:
        missing.append(f"{command} is not on PATH")
if missing:
    raise RuntimeError("Environment setup is incomplete:\n- " + "\n- ".join(missing) + "\nFollow Bonus Unit 1/README.md.")

for directory in (ENVIRONMENT_DIR, HUGGY_LINUX_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"ML-Agents: {version('mlagents')}")
print(f"Notebook workspace: {NOTEBOOK_DIR}")
print(f"Training run ID: {RUN_ID}")


Python: 3.10.12 (/tmp/bonus-unit1-env/bin/python)
ML-Agents: 1.1.0
Notebook workspace: /home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook
Training run ID: Huggy


## Python environment

ML-Agents 1.1.0 requires Python 3.10.1 through 3.10.12. The provided Conda environment pins Python 3.10.12 and installs matching `mlagents` and `mlagents-envs` packages. Keep this notebook kernel attached to that environment; creating a second environment from inside Jupyter does not change the active kernel.


## Download and prepare the Unity environment

The Huggy build is downloaded and extracted under `Notebook/runs/environment/linux/`. This generated environment is local and excluded from Git.


In [3]:
HUGGY_LINUX_DIR.mkdir(parents=True, exist_ok=True)
print(f"Huggy environment directory: {HUGGY_LINUX_DIR}")


Huggy environment directory: /home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/environment/linux


The notebook downloads the prebuilt environment from the [Hugging Face Huggy repository](https://github.com/huggingface/Huggy).


In [4]:
HUGGY_DOWNLOAD_URL = "https://github.com/huggingface/Huggy/raw/main/Huggy.zip"
if not HUGGY_ARCHIVE.exists():
    print(f"Downloading {HUGGY_DOWNLOAD_URL}")
    urllib.request.urlretrieve(HUGGY_DOWNLOAD_URL, HUGGY_ARCHIVE)
print(f"Huggy archive: {HUGGY_ARCHIVE} ({HUGGY_ARCHIVE.stat().st_size:,} bytes)")


Huggy archive: /home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/environment/Huggy.zip (39,214,997 bytes)


In [5]:
if not HUGGY_EXECUTABLE.exists():
    archive_root = HUGGY_LINUX_DIR.resolve()
    with zipfile.ZipFile(HUGGY_ARCHIVE) as archive:
        for member in archive.infolist():
            target = (archive_root / member.filename).resolve()
            if not target.is_relative_to(archive_root):
                raise ValueError(f"Unsafe path in Huggy archive: {member.filename}")
        archive.extractall(archive_root)

if not HUGGY_EXECUTABLE.is_file():
    raise FileNotFoundError(f"Expected Huggy executable at {HUGGY_EXECUTABLE}")
print(f"Extracted Huggy executable: {HUGGY_EXECUTABLE}")


Extracted Huggy executable: /home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/environment/linux/Huggy/Huggy.x86_64


Confirm the downloaded Linux executable is present before starting the trainer.


In [6]:
if not HUGGY_EXECUTABLE.is_file():
    raise FileNotFoundError(f"Missing Huggy executable: {HUGGY_EXECUTABLE}")
HUGGY_EXECUTABLE.chmod(HUGGY_EXECUTABLE.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
print(f"Executable permissions set: {HUGGY_EXECUTABLE}")


Executable permissions set: /home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/environment/linux/Huggy/Huggy.x86_64


## Let's recap how this environment works

### The State Space: what Huggy "perceives."

Huggy doesn't "see" his environment. Instead, we provide him information about the environment:

- The target (stick) position
- The relative position between himself and the target
- The orientation of his legs.

Given all this information, Huggy **can decide which action to take next to fulfill his goal**.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy.jpg" alt="Huggy" width="100%">


### The Action Space: what moves Huggy can do
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy-action.jpg" alt="Huggy action" width="100%">

**Joint motors drive huggy legs**. It means that to get the target, Huggy needs to **learn to rotate the joint motors of each of his legs correctly so he can move**.

### The Reward Function

The reward function is designed so that **Huggy will fulfill his goal** : fetch the stick.

Remember that one of the foundations of Reinforcement Learning is the *reward hypothesis*: a goal can be described as the **maximization of the expected cumulative reward**.

Here, our goal is that Huggy **goes towards the stick but without spinning too much**. Hence, our reward function must translate this goal.

Our reward function:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/reward.jpg" alt="Huggy reward function" width="100%">

- *Orientation bonus*: we **reward him for getting close to the target**.
- *Time penalty*: a fixed-time penalty given at every action to **force him to get to the stick as fast as possible**.
- *Rotation penalty*: we penalize Huggy if **he spins too much and turns too quickly**.
- *Getting to the target reward*: we reward Huggy for **reaching the target**.

## Huggy training configuration

The tracked configuration is [`config/Huggy.yaml`](../config/Huggy.yaml). Edit that file to experiment with PPO hyperparameters, checkpoint frequency, or the training step limit.


In [7]:
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing training configuration: {CONFIG_PATH}")
print(f"Training configuration: {CONFIG_PATH}\n")
print(CONFIG_PATH.read_text())


Training configuration: /home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/config/Huggy.yaml

behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    checkpoint_interval: 200000
    keep_checkpoints: 15
    max_steps: 2000000
    time_horizon: 1000
    summary_freq: 50000



The configuration saves checkpoints every 200,000 steps and keeps up to 15 checkpoints. The default training limit is 2,000,000 steps. Reduce `max_steps` in `config/Huggy.yaml` for a shorter trial.


## Train Huggy

The training cell runs `mlagents-learn` against the downloaded Linux environment. Results are written under `Notebook/runs/results/<run-id>/`. Set `HUGGY_RUN_ID` before launching Jupyter to choose another run name. For a resumed run, set `HUGGY_RESUME=1`.


The course estimates 30–45 minutes with suitable hardware. CPU runs can take substantially longer. The trainer runs without a rendered game window.


In [8]:
train_command = [
    shutil.which("mlagents-learn"),
    str(CONFIG_PATH),
    f"--env={HUGGY_EXECUTABLE}",
    f"--run-id={RUN_ID}",
    f"--results-dir={RESULTS_DIR}",
    "--no-graphics",
]
if os.environ.get("HUGGY_RESUME") == "1":
    train_command.append("--resume")

print("Running:", " ".join(train_command))
subprocess.run(train_command, cwd=NOTEBOOK_DIR, check=True)


Running: /tmp/bonus-unit1-env/bin/mlagents-learn /home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/config/Huggy.yaml --env=/home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/environment/linux/Huggy/Huggy.x86_64 --run-id=Huggy --results-dir=/home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/results --no-graphics


/tmp/bonus-unit1-env/lib/python3.10/site-packages/mlagents/torch_utils/torch.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



            ┐  ╖
        ╓╖╬│╡  ││╬╖╖
    ╓╖╬│││││┘  ╬│││││╬╖
 ╖╬│││││╬╜        ╙╬│││││╖╖                               ╗╗╗
 ╬╬╬╬╖││╦╖        ╖╬││╗╣╣╣╬      ╟╣╣╬    ╟╣╣╣             ╜╜╜  ╟╣╣
 ╬╬╬╬╬╬╬╬╖│╬╖╖╓╬╪│╓╣╣╣╣╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╒╣╣╖╗╣╣╣╗   ╣╣╣ ╣╣╣╣╣╣ ╟╣╣╖   ╣╣╣
 ╬╬╬╬┐  ╙╬╬╬╬│╓╣╣╣╝╜  ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╣╙ ╙╣╣╣  ╣╣╣ ╙╟╣╣╜╙  ╫╣╣  ╟╣╣
 ╬╬╬╬┐     ╙╬╬╣╣      ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣     ╣╣╣┌╣╣╜
 ╬╬╬╜       ╬╬╣╣      ╙╝╣╣╬      ╙╣╣╣╗╖╓╗╣╣╣�� ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣╦╓    ╣╣╣╣╣
 ╙   ╓╦╖    ╬╬╣╣   ╓╗╗╖            ╙╝╣╣╣╣╝╜   ╘╝╝╜   ╝╝╝  ╝╝╝   ╙╣╣╣    ╟╣╣╣
   ╩╬╬╬╬╬╬╦╦╬╬╣╣╗╣╣╣╣╣╣╣╝                                             ╫╣╣╣╣
      ╙╬╬╬╬╬╬╬╣╣╣╣╣╣╝╜
          ╙╬╬╬╣╣╣╜
             ╙
        
 Version information:
  ml-agents: 1.1.0,
  ml-agents-envs: 1.1.0,
  Communicator API: 1.5.0,
  PyTorch: 2.2.1+cpu


[INFO] Connected to Unity environment with package version 2.2.1-exp.1 and communication version 1.5.0
[INFO] Connected new brain: Huggy?team=0
[INFO] Hyperparameters for behavior name Huggy: 
	trainer_type:	ppo
	hyperparameters:	
	  batch_size:	2048
	  buffer_size:	20480
	  learning_rate:	0.0003
	  beta:	0.005
	  epsilon:	0.2
	  lambd:	0.95
	  num_epoch:	3
	  shared_critic:	False
	  learning_rate_schedule:	linear
	  beta_schedule:	linear
	  epsilon_schedule:	linear
	checkpoint_interval:	200000
	network_settings:	
	  normalize:	True
	  hidden_units:	512
	  num_layers:	3
	  vis_encode_type:	simple
	  memory:	None
	  goal_conditioning_type:	hyper
	  deterministic:	False
	reward_signals:	
	  extrinsic:	
	    gamma:	0.995
	    strength:	1.0
	    network_settings:	
	      normalize:	False
	      hidden_units:	128
	      num_layers:	2
	      vis_encode_type:	simple
	      memory:	None
	      goal_conditioning_type:	hyper
	      deterministic:	False
	init_path:	None
	keep_checkpoints:	15
	eve

[INFO] Huggy. Step: 50000. Time Elapsed: 37.783 s. Mean Reward: 1.847. Std of Reward: 0.776. Training.


...

[INFO] Huggy. Step: 1850000. Time Elapsed: 1480.842 s. Mean Reward: 3.570. Std of Reward: 2.209. Training.


[INFO] Huggy. Step: 1900000. Time Elapsed: 1523.215 s. Mean Reward: 3.655. Std of Reward: 2.178. Training.


[INFO] Huggy. Step: 1950000. Time Elapsed: 1561.629 s. Mean Reward: 3.586. Std of Reward: 2.186. Training.


[INFO] Huggy. Step: 2000000. Time Elapsed: 1601.089 s. Mean Reward: 3.504. Std of Reward: 2.180. Training.
[INFO] Exported /home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/results/Huggy/Huggy/Huggy-1999882.onnx
[INFO] Exported /home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/results/Huggy/Huggy/Huggy-2000003.onnx
[INFO] Copied /home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/results/Huggy/Huggy/Huggy-2000003.onnx to /home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/results/Huggy/Huggy.onnx.


CompletedProcess(args=['/tmp/bonus-unit1-env/bin/mlagents-learn', '/home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/config/Huggy.yaml', '--env=/home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/environment/linux/Huggy/Huggy.x86_64', '--run-id=Huggy', '--results-dir=/home/obidit/Coding/ObiLearnsRL/Bonus Unit 1/Notebook/runs/results', '--no-graphics'], returncode=0)

## Optional: upload the trained agent to Hugging Face Hub

Uploading is skipped unless both `HF_TOKEN` and `HF_REPO_ID` are set in the environment before launching Jupyter. The final cell publishes the trained run when those values are present.


To publish the model, create a Hugging Face token with write access and set the two environment variables in your shell before starting Jupyter:

```bash
export HF_TOKEN=hf_...
export HF_REPO_ID=your-username/your-huggy-repo
```

The token stays outside the notebook file.


In [9]:
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_REPO_ID = os.environ.get("HF_REPO_ID")
if HF_TOKEN and HF_REPO_ID:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f"Hub upload enabled for {HF_REPO_ID}.")
else:
    print("Hub upload is disabled. Set HF_TOKEN and HF_REPO_ID before launching Jupyter to enable it.")


Hub upload is disabled. Set HF_TOKEN and HF_REPO_ID before launching Jupyter to enable it.


The upload command uses the active environment's `mlagents-push-to-hf` tool; no interactive notebook login is needed.


The final cell only uploads when both Hub environment variables are configured.


The command uses the training run ID, its results directory, your repository ID, and a commit message. If the repository does not exist, ML-Agents can create it for you.


In [10]:
if not HF_TOKEN or not HF_REPO_ID:
    print("Skipped Hub upload. Set HF_TOKEN and HF_REPO_ID before launching Jupyter to publish this run.")
else:
    run_directory = RESULTS_DIR / RUN_ID
    if not run_directory.is_dir():
        raise FileNotFoundError(f"Training output not found: {run_directory}. Run the training cell first.")
    push_command = [
        shutil.which("mlagents-push-to-hf"),
        f"--run-id={RUN_ID}",
        f"--local-dir={run_directory}",
        f"--repo-id={HF_REPO_ID}",
        "--commit-message=Train Huggy with PPO",
    ]
    print("Running:", " ".join(push_command))
    subprocess.run(push_command, cwd=NOTEBOOK_DIR, check=True)


Skipped Hub upload. Set HF_TOKEN and HF_REPO_ID before launching Jupyter to publish this run.


If upload succeeds, the command prints the Hugging Face repository URL. The repository contains the exported model and training artifacts.


But now comes the best: **being able to play with Huggy online 👀.**

## Play with your Huggy 🐕

This step is the simplest:

- Open the game Huggy in your browser: https://huggingface.co/spaces/ThomasSimonini/Huggy

- Click on Play with my Huggy model

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/load-huggy.jpg" alt="load-huggy" width="100%">

1. In step 1, type your username (your username is case sensitive: for instance, my username is ThomasSimonini not thomassimonini or ThOmasImoNInI) and click on the search button.

2. In step 2, select your model repository.

3. In step 3, **choose which model you want to replay**:
  - I have multiple ones, since we saved a model every 500000 timesteps.
  - But since I want the more recent, I choose `Huggy.onnx`

👉 What’s nice **is to try with different models steps to see the improvement of the agent.**

Congrats on finishing this bonus unit!

You can now sit and enjoy playing with your Huggy 🐶. And don't **forget to spread the love by sharing Huggy with your friends 🤗**. And if you share about it on social media, **please tag us @huggingface and me @simoninithomas**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy-cover.jpeg" alt="Huggy cover" width="100%">


## Keep Learning, Stay  awesome 🤗

## Saved execution results

This notebook completed the configured 2,000,000-step training run. The setup and training outputs above are saved in this notebook.

- Local final model: [`Huggy.onnx`](../runs/results/Huggy/Huggy.onnx)
- Published model: [`tashobi02/ppo-huggy`](https://huggingface.co/tashobi02/ppo-huggy)

The optional upload cells were skipped during notebook execution because no `HF_REPO_ID` was set; the completed run was published afterward.